# API Examples — the training pipeline & model in action

Runnable companion to [`docs/API_EXAMPLES.md`](../docs/API_EXAMPLES.md). (The Streamlit UIs aren't imported; we use `train_model.py` + the saved model.)

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
for p in ('.', 'src'):
    sys.path.insert(0, os.path.abspath(p))

## train_model.py — config + cleaning

In [2]:
import pandas as pd
import train_model

print('price_unit:', train_model.CONFIG['price_unit'])

# clean_dataset trims/normalises strings and drops the invalid (negative-price) row
df = pd.DataFrame({'selling_price': [5.0, -1.0],
                   'make': [' maruti ', 'x'],
                   'model': ['swift vxi', 'y']})
train_model.clean_dataset(df)[['make', 'model', 'selling_price']]

price_unit: Lakhs
-> Dropped 1 rows with missing/invalid values (50.0%)


,make,model,selling_price
0,MARUTI,SWIFT VXI,5.0


## Price bands + slider bounds (from the full dataset)

In [3]:
full = train_model.clean_dataset(train_model.load_dataset())

_, range_config = train_model.make_price_ranges(full)
print('band edges:', range_config['bin_edges'])
train_model.numeric_input_specs(full)['age']

-> Loaded 19,820 rows x 17 columns from data/cars24-car-price-cleaned-new.csv.gz
-> Price-range distribution:
     Low      6,618 cars (33.4%)
     Medium   6,757 cars (34.1%)
     High     6,445 cars (32.5%)
band edges: [0.3, 3.99, 6.75, 20.9]


{'min': 3, 'max': 17, 'default': 8, 'step': 1, 'dtype': 'int'}

## Predict with the saved model (price is in **Lakhs**)

In [4]:
import json, joblib
from pathlib import Path

MODELS = Path(train_model.__file__).parent / 'models'
model = joblib.load(MODELS / 'price_model.pkl')
cols = json.loads((MODELS / 'feature_columns.json').read_text())

# Build one fully-specified feature row (one-hot columns default to 0)
row = {c: 0 for c in cols}
row.update({'km_driven': 40000, 'mileage': 20.0, 'engine': 1200, 'max_power': 82.0, 'age': 5,
            'make': 'MARUTI', 'model': 'SWIFT VXI', 'Petrol': 1, 'Manual': 1, 'Seats_5': 1})

price = float(model.predict(pd.DataFrame([row])[cols])[0])
print(f'Rs {price:.2f} Lakhs')

Rs 5.21 Lakhs


A Swift lands around a few Lakhs (not ₹500000) — see `docs/API_EXAMPLES.md`.